# In-class: SARSA, Expected SARSA and Q-learning on the Toy MDP

**ESE 561.** The loops that run the agent and print every step are given.

The toy MDP: states $A,B$, terminal $G$, $\gamma=0.9$.

| $(s,a)$ | next state, probability, reward |
|---|---|
| $(A,\text{stay})$ | $A$, 1, $+1$ |
| $(A,\text{go})$ | $B$, 0.8, 0;  $A$, 0.2, 0 |
| $(B,\text{cash})$ | $G$, 0.7, $+20$;  $A$, 0.3, 0 |
| $(B,\text{back})$ | $A$, 1, 0 |

In [1]:
import numpy as np

P = {('A', 'stay'): [('A', 1.0, 1.0)],
     ('A', 'go'):   [('B', 0.8, 0.0), ('A', 0.2, 0.0)],
     ('B', 'cash'): [('G', 0.7, 20.0), ('A', 0.3, 0.0)],
     ('B', 'back'): [('A', 1.0, 0.0)]}
ACTIONS = {'A': ['stay', 'go'], 'B': ['cash', 'back']}
PAIRS = list(P)
gamma = 0.9

def env_step(s, a, rng):
    """The environment: sample s' ~ p(.|s, a). Returns (s', r). The agent never sees P."""
    outcomes = P[(s, a)]
    k = rng.choice(len(outcomes), p=[p for _, p, _ in outcomes])
    s2, _, r = outcomes[k]
    return s2, r

def check(name, got, want, tol=1e-9):
    try:
        ok = (got == want) if isinstance(want, (str, dict)) else abs(got - want) < tol
    except TypeError:
        ok = False
    print(f'{name}: OK' if ok else f'{name}: expected {want}, got {got}')

## 1. The Q-table and the policy as dictionaries

`Q[(s, a)]` is a number. `pi[s][a]` is the probability of taking $a$ in $s$. The ε-greedy policy gives the greedy action $1-\varepsilon+\varepsilon/n$ and every other action $\varepsilon/n$, where $n$ is the number of actions in $s$.

**Exercise 1.** Return the greedy action: the action in `ACTIONS[s]` with the largest `Q[(s, a)]`. (Hint: `max` with `key=`.)

**Exercise 2.** Fill in the probability `pi[s][a]`; the greedy action is `g`.

In [2]:
def greedy(Q, s):
    return max(ACTIONS[s], key=lambda a: Q[(s, a)])

def eps_greedy_policy(Q, eps):
    pi = {}
    for s in ACTIONS:
        g, n = greedy(Q, s), len(ACTIONS[s])
        pi[s] = {}
        for a in ACTIONS[s]:
            pi[s][a] = 1 - eps + eps / n if a == g else eps / n
    return pi

def sample_action(pi, s, rng):
    acts = list(pi[s])
    return acts[rng.choice(len(acts), p=[pi[s][a] for a in acts])]

In [3]:
Q_test = {('A', 'stay'): 1.0, ('A', 'go'): 2.0, ('B', 'cash'): 3.0, ('B', 'back'): 4.0}
check('Exercise 1', greedy(Q_test, 'B'), 'back')
check('Exercise 2', eps_greedy_policy(Q_test, 0.3), {'A': {'stay': 0.15, 'go': 0.85}, 'B': {'cash': 0.15, 'back': 0.85}})

Exercise 1: OK
Exercise 2: OK


**Q1.** In $B$, with $\varepsilon=0.3$, what is the probability that the agent takes `back` when `cash` is greedy? Why is it not $0.3$?

**Answer.** 

## 2. One update

All three methods move one entry of the table towards a target $Y$:
$$Q(s,a)\leftarrow Q(s,a)+\alpha\,\bigl(Y-Q(s,a)\bigr).$$

**Exercise 3.** Write this update.

In [4]:
def td_update(Q, s, a, target, alpha):
    Q[(s, a)] += alpha * (target - Q[(s, a)])

In [5]:
Q_tmp = dict(Q_test); td_update(Q_tmp, 'A', 'go', 12.0, 0.1)
check('Exercise 3', Q_tmp[('A', 'go')], 3.0)
check('Exercise 3, other entries unchanged', sum(Q_tmp.values()) - Q_tmp[('A', 'go')], 8.0)

Exercise 3: OK
Exercise 3, other entries unchanged: OK


## 3. The three targets

After the step $(s,a)\to(r,s')$ the agent also samples its next action $a'\sim\pi(\cdot\mid s')$, because it has to act. The targets differ only in how they value $s'$:

| method | target $Y$ | uses the sampled $a'$? |
|---|---|---|
| SARSA | $r+\gamma\,Q(s',a')$ | yes |
| Expected SARSA | $r+\gamma\sum_b\pi(b\mid s')\,Q(s',b)$ | no, averages over it |
| Q-learning | $r+\gamma\max_bQ(s',b)$ | no |

(If $s'=G$ the target is just $r$; the loop handles that case.)

**Exercises 4–6.** Write the three targets. All have the same arguments, so the loop can call any of them.

In [6]:
def sarsa_target(r, s2, a2, Q, pi):
    return r + gamma * Q[(s2, a2)]

def q_learning_target(r, s2, a2, Q, pi):
    return r + gamma * max(Q[(s2, b)] for b in ACTIONS[s2])

def expected_sarsa_target(r, s2, a2, Q, pi):
    return r + gamma * sum(pi[s2][b] * Q[(s2, b)] for b in ACTIONS[s2])

In [7]:
pi_test = eps_greedy_policy(Q_test, 0.3)     # in B: cash 0.15, back 0.85
check('Exercise 4', sarsa_target(0.0, 'B', 'cash', Q_test, pi_test), 0.9 * 3)
check('Exercise 5', q_learning_target(0.0, 'B', 'cash', Q_test, pi_test), 0.9 * 4)
check('Exercise 6', expected_sarsa_target(0.0, 'B', 'cash', Q_test, pi_test), 0.9 * (0.15 * 3 + 0.85 * 4))

Exercise 4: OK
Exercise 5: OK
Exercise 6: OK


**Q2.** In the check, the sampled next action is `cash` but the greedy one is `back`. Which of the three targets changes if $a'$ is changed to `back`?

**Answer.** 

## 4. The loop (given): run it step by step

`td_control` plays episodes: each starts in $A$ and ends when the agent reaches $G$. Inside an episode, at every step it prints what was **sampled** ($a$ by the agent, $s'$ and $r$ by the environment, then $a'$ by the agent), the target, and the one entry that changes. The step counter `t` runs across episodes.

Unlike Monte Carlo, TD does **not** wait for the end of the episode: $Q$ is updated after every step, and the policy is recomputed from the new $Q$ right away, so later steps of the same episode already use what was just learned. The end of an episode only means: the target is $r$ (no next state), and the next step restarts in $A$.

In [8]:
def show_Q(Q):
    return '  '.join(f'{s},{a}={Q[(s, a)]:5.2f}' for s, a in PAIRS)

def td_control(target_fn, n_episodes, eps=0.3, alpha=0.1, seed=0, verbose=3):
    """Play n_episodes episodes; print every step of the first `verbose` episodes."""
    rng = np.random.default_rng(seed)
    Q = {sa: 0.0 for sa in PAIRS}
    t = 0                                                             # total step counter, for printing
    for episode in range(n_episodes):
        pi = eps_greedy_policy(Q, eps)
        s = 'A'; a = sample_action(pi, s, rng)                        # every episode starts in A
        G, k = 0.0, 0                                                 # discounted return, length
        if episode < verbose: print(f'=== episode {episode}')
        while s != 'G':
            s2, r = env_step(s, a, rng)                               # environment samples s', r
            if s2 == 'G':
                a2, target = None, r
            else:
                a2 = sample_action(pi, s2, rng)                       # agent samples a' (to act next)
                target = target_fn(r, s2, a2, Q, pi)
            nxt = ('terminal' if s2 == 'G' else                       # what the target saw (before the update)
                   f"a'={a2}  Q({s2},.)={{{', '.join(f'{b}: {Q[(s2, b)]:.2f}' for b in ACTIONS[s2])}}}"
                   f"  pi(.|{s2})={{{', '.join(f'{b}: {pi[s2][b]:.2f}' for b in ACTIONS[s2])}}}")
            old = Q[(s, a)]
            td_update(Q, s, a, target, alpha)                         # update INSIDE the episode
            if episode < verbose:
                print(f't={t:2d} ({s},{a:4s}) -> s\'={s2}, r={r:2.0f}  {nxt}')
                print(f'       Y={target:5.2f}   Q({s},{a}): {old:.2f} -> {Q[(s, a)]:.2f}    [{show_Q(Q)}]')
            G += gamma ** k * r; k += 1; t += 1
            pi = eps_greedy_policy(Q, eps)                            # the policy follows the new Q
            s, a = s2, a2                                             # the sampled a' is taken
        if episode < verbose: print(f'--- episode {episode} ended after {k} steps, discounted return {G:.2f}\n')
    return Q

print('SARSA'); Q1 = td_control(sarsa_target, 3)

SARSA
=== episode 0
t= 0 (A,stay) -> s'=A, r= 1  a'=stay  Q(A,.)={stay: 0.00, go: 0.00}  pi(.|A)={stay: 0.85, go: 0.15}
       Y= 1.00   Q(A,stay): 0.00 -> 0.10    [A,stay= 0.10  A,go= 0.00  B,cash= 0.00  B,back= 0.00]
t= 1 (A,stay) -> s'=A, r= 1  a'=stay  Q(A,.)={stay: 0.10, go: 0.00}  pi(.|A)={stay: 0.85, go: 0.15}
       Y= 1.09   Q(A,stay): 0.10 -> 0.20    [A,stay= 0.20  A,go= 0.00  B,cash= 0.00  B,back= 0.00]
t= 2 (A,stay) -> s'=A, r= 1  a'=stay  Q(A,.)={stay: 0.20, go: 0.00}  pi(.|A)={stay: 0.85, go: 0.15}
       Y= 1.18   Q(A,stay): 0.20 -> 0.30    [A,stay= 0.30  A,go= 0.00  B,cash= 0.00  B,back= 0.00]
t= 3 (A,stay) -> s'=A, r= 1  a'=stay  Q(A,.)={stay: 0.30, go: 0.00}  pi(.|A)={stay: 0.85, go: 0.15}
       Y= 1.27   Q(A,stay): 0.30 -> 0.39    [A,stay= 0.39  A,go= 0.00  B,cash= 0.00  B,back= 0.00]
t= 4 (A,stay) -> s'=A, r= 1  a'=stay  Q(A,.)={stay: 0.39, go: 0.00}  pi(.|A)={stay: 0.85, go: 0.15}
       Y= 1.35   Q(A,stay): 0.39 -> 0.49    [A,stay= 0.49  A,go= 0.00  B,cash= 0.00 

In [9]:
print('Q-learning'); Q2 = td_control(q_learning_target, 3)

Q-learning
=== episode 0
t= 0 (A,stay) -> s'=A, r= 1  a'=stay  Q(A,.)={stay: 0.00, go: 0.00}  pi(.|A)={stay: 0.85, go: 0.15}
       Y= 1.00   Q(A,stay): 0.00 -> 0.10    [A,stay= 0.10  A,go= 0.00  B,cash= 0.00  B,back= 0.00]
t= 1 (A,stay) -> s'=A, r= 1  a'=stay  Q(A,.)={stay: 0.10, go: 0.00}  pi(.|A)={stay: 0.85, go: 0.15}
       Y= 1.09   Q(A,stay): 0.10 -> 0.20    [A,stay= 0.20  A,go= 0.00  B,cash= 0.00  B,back= 0.00]
t= 2 (A,stay) -> s'=A, r= 1  a'=stay  Q(A,.)={stay: 0.20, go: 0.00}  pi(.|A)={stay: 0.85, go: 0.15}
       Y= 1.18   Q(A,stay): 0.20 -> 0.30    [A,stay= 0.30  A,go= 0.00  B,cash= 0.00  B,back= 0.00]
t= 3 (A,stay) -> s'=A, r= 1  a'=stay  Q(A,.)={stay: 0.30, go: 0.00}  pi(.|A)={stay: 0.85, go: 0.15}
       Y= 1.27   Q(A,stay): 0.30 -> 0.39    [A,stay= 0.39  A,go= 0.00  B,cash= 0.00  B,back= 0.00]
t= 4 (A,stay) -> s'=A, r= 1  a'=stay  Q(A,.)={stay: 0.39, go: 0.00}  pi(.|A)={stay: 0.85, go: 0.15}
       Y= 1.35   Q(A,stay): 0.39 -> 0.49    [A,stay= 0.49  A,go= 0.00  B,cash= 

In [10]:
print('Expected SARSA'); Q3 = td_control(expected_sarsa_target, 3)

Expected SARSA
=== episode 0
t= 0 (A,stay) -> s'=A, r= 1  a'=stay  Q(A,.)={stay: 0.00, go: 0.00}  pi(.|A)={stay: 0.85, go: 0.15}
       Y= 1.00   Q(A,stay): 0.00 -> 0.10    [A,stay= 0.10  A,go= 0.00  B,cash= 0.00  B,back= 0.00]
t= 1 (A,stay) -> s'=A, r= 1  a'=stay  Q(A,.)={stay: 0.10, go: 0.00}  pi(.|A)={stay: 0.85, go: 0.15}
       Y= 1.08   Q(A,stay): 0.10 -> 0.20    [A,stay= 0.20  A,go= 0.00  B,cash= 0.00  B,back= 0.00]
t= 2 (A,stay) -> s'=A, r= 1  a'=stay  Q(A,.)={stay: 0.20, go: 0.00}  pi(.|A)={stay: 0.85, go: 0.15}
       Y= 1.15   Q(A,stay): 0.20 -> 0.29    [A,stay= 0.29  A,go= 0.00  B,cash= 0.00  B,back= 0.00]
t= 3 (A,stay) -> s'=A, r= 1  a'=stay  Q(A,.)={stay: 0.29, go: 0.00}  pi(.|A)={stay: 0.85, go: 0.15}
       Y= 1.22   Q(A,stay): 0.29 -> 0.39    [A,stay= 0.39  A,go= 0.00  B,cash= 0.00  B,back= 0.00]
t= 4 (A,stay) -> s'=A, r= 1  a'=stay  Q(A,.)={stay: 0.39, go: 0.00}  pi(.|A)={stay: 0.85, go: 0.15}
       Y= 1.30   Q(A,stay): 0.39 -> 0.48    [A,stay= 0.48  A,go= 0.00  B,ca

**Q3.** The second line of each step shows $Q(s',\cdot)$ and $\pi(\cdot\mid s')$ **as the target saw them**. With the same seed the three runs sample the same transitions at first.
(a) At which step do the SARSA and Q-learning targets first differ? Recompute both targets at that step by hand.
(b) Expected SARSA's target differs from the others already at $t=1$, although $a'$ is greedy there. Recompute it by hand.

**Answer.** 

**Q4.** All three runs sample $a'$. Why does Q-learning need it, if its target does not use it?

**Answer.** 

**Q4b.** In episode 0, at which step does $Q(B,\text{cash})$ first become nonzero? Does any later step of the same episode use it? Would Monte Carlo have updated anything before the end of episode 0?

**Answer.** At $t=7$, the last step of episode 0 ($B$, cash $\to G$, $Y=20$). Episode 0 ends there, so it is first used at $t=8$ in episode 1 ($Y=0.9\cdot2.00=1.80$ for $(A,\text{go})$). Within episode 0, however, $Q(A,\text{stay})$ was updated at $t=0$ and used already at $t=1$. Monte Carlo would have updated nothing until $t=7$, and would then have used the actual return of the whole episode.

## 5. How noisy are the targets?

Fix $Q=Q^*$ and the ε-greedy policy, and look at the target for the pair $(A,\text{go})$ over many random $(s',a')$. Everything below is given.

In [11]:
def exact(kind, eps=0.3, K=1000):
    """Simple iterations with the model: kind='max' gives Q*, kind='eps' gives Q of the eps-greedy policy."""
    Q = {sa: 0.0 for sa in PAIRS}
    for _ in range(K):
        pi = eps_greedy_policy(Q, eps)
        v = lambda s2: 0.0 if s2 == 'G' else (max(Q[(s2, b)] for b in ACTIONS[s2]) if kind == 'max'
                                              else sum(pi[s2][b] * Q[(s2, b)] for b in ACTIONS[s2]))
        Q = {(s, a): sum(p * (r + gamma * v(s2)) for s2, p, r in P[(s, a)]) for (s, a) in PAIRS}
    return Q

Q_star, Q_eps = exact('max'), exact('eps')
rng = np.random.default_rng(1)
pi_star = eps_greedy_policy(Q_star, 0.3)
for name, fn in [('SARSA', sarsa_target), ('Expected SARSA', expected_sarsa_target), ('Q-learning', q_learning_target)]:
    Ys = []
    for _ in range(100_000):
        s2, r = env_step('A', 'go', rng)
        Ys.append(fn(r, s2, sample_action(pi_star, s2, rng), Q_star, pi_star))
    print(f'{name:15s} mean {np.mean(Ys):6.2f}   std {np.std(Ys):5.2f}')

SARSA           mean  15.68   std  1.28
Expected SARSA  mean  15.68   std  0.63
Q-learning      mean  16.11   std  0.81


**Q5.** Which two methods have the same mean target? Why is the spread of Expected SARSA smaller than that of SARSA, and where does the remaining spread come from?

**Answer.** 

## 6. Q-learning with a uniform sweep (deterministic, no behavior policy)

With a simulator we can update **every** pair in a fixed order, sweep after sweep. The only random draw is the environment's. The step size decreases with the number $n(s,a)$ of updates of that pair: $\alpha=n^{-0.7}$ (Robbins–Monro).

**Exercise 7.** The step size.

**Exercise 8.** After the sweeps, the deterministic greedy policy as a dictionary `{state: action}`.

In [12]:
def q_learning_sweep(target_fn, n_sweeps, seed=0, verbose=1):
    rng = np.random.default_rng(seed)
    Q = {sa: 0.0 for sa in PAIRS}; n = {sa: 0 for sa in PAIRS}
    for k in range(n_sweeps):
        for (s, a) in PAIRS:                                          # chosen by the sweep, not by a policy
            s2, r = env_step(s, a, rng)
            n[(s, a)] += 1
            alpha = n[(s, a)] ** -0.7
            target = r if s2 == 'G' else target_fn(r, s2, None, Q, eps_greedy_policy(Q, 0.3))
            old = Q[(s, a)]
            td_update(Q, s, a, target, alpha)
            if k < verbose:
                print(f'({s},{a:4s}) -> s\'={s2}, r={r:2.0f}  alpha={alpha:.2f}  Y={target:5.2f}  Q: {old:.2f} -> {Q[(s, a)]:.2f}')
    return Q

Q_sweep = q_learning_sweep(q_learning_target, 5000)
pi_det = {s: greedy(Q_sweep, s) for s in ACTIONS}
print('\n' + show_Q(Q_sweep)); print('greedy policy:', pi_det)

(A,stay) -> s'=A, r= 1  alpha=1.00  Y= 1.00  Q: 0.00 -> 1.00
(A,go  ) -> s'=B, r= 0  alpha=1.00  Y= 0.00  Q: 0.00 -> 0.00
(B,cash) -> s'=G, r=20  alpha=1.00  Y=20.00  Q: 0.00 -> 20.00
(B,back) -> s'=A, r= 0  alpha=1.00  Y= 0.90  Q: 0.00 -> 0.90

A,stay=15.55  A,go=16.09  B,cash=18.33  B,back=14.55
greedy policy: {'A': 'go', 'B': 'cash'}


In [13]:
check('Exercise 7, final values', round(Q_sweep[('A', 'go')], 1), round(Q_star[('A', 'go')], 1), tol=0.15)
check('Exercise 8', pi_det, {'A': 'go', 'B': 'cash'})

Exercise 7, final values: OK
Exercise 8: OK


**Q6.** The sweep passes `a2=None` to the target. Why does that not break Q-learning? Which target from Section 3 could **not** be used here?

**Answer.** 

**Q7.** We change the argument to run the sweep with `expected_sarsa_target`. To which table does it converge, and why? Try it.

In [14]:
Q_sweep_es = q_learning_sweep(expected_sarsa_target, 5000, verbose=0)
print(show_Q(Q_sweep_es)); print(show_Q(Q_eps))

A,stay=14.74  A,go=15.28  B,cash=18.08  B,back=13.74
A,stay=14.68  A,go=15.29  B,cash=18.10  B,back=13.68


**Answer.**  

(Expected SARSA can therefore also be used off-policy; SARSA cannot.)

## 7. Long runs: who converges where (given)

In [ ]:
runs = {'SARSA':          [td_control(sarsa_target, 5000, seed=i, verbose=0) for i in range(10)],
        'Expected SARSA': [td_control(expected_sarsa_target, 5000, seed=i, verbose=0) for i in range(10)],
        'Q-learning':     [td_control(q_learning_target, 5000, seed=i, verbose=0) for i in range(10)],
        'Q-l. sweep':     [q_learning_sweep(q_learning_target, 5000, seed=i, verbose=0) for i in range(10)]}
print(f'{"":16s}' + ''.join(f'{s + "," + a:>9s}' for s, a in PAIRS))
print(f'{"exact Q*":16s}' + ''.join(f'{Q_star[k]:9.2f}' for k in PAIRS))
print(f'{"exact Q^pi_eps":16s}' + ''.join(f'{Q_eps[k]:9.2f}' for k in PAIRS))
for name, Qs in runs.items():
    print(f'{name:16s}' + ''.join(f'{np.mean([q[k] for q in Qs]):9.2f}' for k in PAIRS))

**Q8.** Which rows are close to $Q^*$ and which to $Q^{\pi_\varepsilon}$? Explain from the targets alone.

**Answer.** 

**Q9.** Do all methods find the same greedy policy? Then why might one prefer SARSA's values in a real environment (think of the cliff)?

**Answer.** 

**Q10.** In the ε-greedy runs (5000 episodes each) α = 0.1 is constant. What would you change to make the SARSA row converge exactly, and what would you change to make it converge to $Q^*$?

**Answer.** Exactly: a decreasing step size per pair, $\sum\alpha=\infty$, $\sum\alpha^2<\infty$ (e.g. $n^{-0.7}$), as in the sweep. To $Q^*$: let ε → 0 slowly, e.g. $\varepsilon_n=1/\sqrt n$ (GLIE). Then the policy SARSA evaluates becomes greedy.